# Internet Requirement Scan — `harbor_4.8opus_tasks_v3`

Scans all task `instruction.md` files (and optionally Dockerfiles) to identify tasks that require internet access during the **agent session** (not just at build time).

**Read-only — does not modify any task files.**

In [1]:
from pathlib import Path
import re
import tomllib
import pandas as pd

DATASET_DIR = Path("/home/ec2-user/endless-terminals-playground/harbor_4.8opus_tasks_v3")

## 1. Keyword patterns

Two tiers:
- **High confidence**: very direct signals (URL in instruction, `curl`/`wget` to external host, `git clone http`, `pip install` at runtime, API call to external service)
- **Medium confidence**: softer signals that might imply internet use ("fetch", "pull from", "remote server", "sync from")

In [2]:
HIGH_PATTERNS = [
    (r'https?://', 'URL in instruction'),
    (r'\bcurl\b.{0,60}http', 'curl to URL'),
    (r'\bwget\b.{0,60}http', 'wget URL'),
    (r'git clone.{0,60}http', 'git clone remote'),
    (r'git clone.{0,60}git@', 'git clone SSH remote'),
    (r'pip install(?! -r|\s+\.)', 'pip install package (runtime)'),
    (r'apt.?get install', 'apt-get install (runtime)'),
    (r'npm install', 'npm install (runtime)'),
    (r'download.{0,40}(file|package|from|at)', 'download from external'),
    (r'(fetch|pull).{0,30}(remote|external|internet|online|server|url|endpoint)', 'fetch/pull from remote'),
    (r'api.{0,20}(key|token|endpoint|call|request).{0,30}(external|internet|online|service)', 'external API call'),
    (r'internet.{0,20}(access|connection|connect)', 'explicit internet reference'),
    (r'(connect|connecting).{0,30}(internet|online|remote host|external)', 'connect to internet/remote'),
    (r'\brsync.{0,60}(remote|@|::\w)', 'rsync to remote host'),
    (r'\bscp\b.{0,60}@', 'scp to remote host'),
    (r'\bsftp\b.{0,60}@', 'sftp to remote host'),
    (r'(push|pull).{0,30}(to|from).{0,30}(remote|origin|upstream)', 'git push/pull remote'),
]

MEDIUM_PATTERNS = [
    (r'\bdownload\b', 'download (generic)'),
    (r'\bfetch\b', 'fetch (generic)'),
    (r'\bremote\b.{0,30}(server|host|endpoint)', 'remote server reference'),
    (r'\bsync\b.{0,30}(from|with).{0,30}(remote|external|server)', 'sync from remote'),
    (r'\bcheck.{0,20}(connectivity|internet|network|online)', 'check connectivity'),
    (r'\b(upload|post).{0,30}(to|endpoint|api|service)', 'upload/post to service'),
]

def find_matches(text, patterns):
    """Return list of (reason, snippet) for all matching patterns."""
    text_lower = text.lower()
    hits = []
    for pattern, reason in patterns:
        m = re.search(pattern, text_lower)
        if m:
            start = max(0, m.start() - 20)
            end = min(len(text), m.end() + 60)
            snippet = text[start:end].replace('\n', ' ').strip()
            hits.append((reason, snippet))
    return hits

## 2. Scan all tasks

In [3]:
task_dirs = sorted(d for d in DATASET_DIR.iterdir() if d.is_dir())
print(f"Total tasks: {len(task_dirs)}")

Total tasks: 8218


In [4]:
results = []

for task_dir in task_dirs:
    instruction_path = task_dir / "instruction.md"
    toml_path = task_dir / "task.toml"
    dockerfile_path = task_dir / "environment" / "Dockerfile"

    if not instruction_path.exists():
        continue

    instruction = instruction_path.read_text(errors="replace")

    allow_internet = True  # harbor default
    category = ""
    if toml_path.exists():
        try:
            cfg = tomllib.loads(toml_path.read_text())
            allow_internet = cfg.get("environment", {}).get("allow_internet", True)
            category = cfg.get("metadata", {}).get("category", "")
        except Exception:
            pass

    high_hits = find_matches(instruction, HIGH_PATTERNS)
    med_hits  = find_matches(instruction, MEDIUM_PATTERNS)

    # Extra check: instruction mentions install but Dockerfile doesn't bake it in
    dockerfile_note = ""
    if dockerfile_path.exists():
        df_text = dockerfile_path.read_text(errors="replace")
        if re.search(r'pip install|apt.?get install', instruction.lower()):
            if not re.search(r'pip install|apt.?get install', df_text.lower()):
                dockerfile_note = "instruction mentions install but Dockerfile does not"

    if high_hits or med_hits or dockerfile_note:
        confidence = "high" if high_hits else "medium"
        all_reasons = [r for r, _ in high_hits] + [r for r, _ in med_hits]
        if dockerfile_note:
            all_reasons.append(dockerfile_note)
        all_snippets = [s for _, s in high_hits] + [s for _, s in med_hits]
        results.append({
            "task": task_dir.name,
            "category": category,
            "allow_internet_current": allow_internet,
            "confidence": confidence,
            "reasons": "; ".join(dict.fromkeys(all_reasons)),
            "first_snippet": all_snippets[0] if all_snippets else "",
            "n_high_signals": len(high_hits),
            "n_medium_signals": len(med_hits),
        })

df = pd.DataFrame(results)
print(f"Flagged tasks: {len(df)} / {len(task_dirs)} ({100*len(df)/len(task_dirs):.1f}%)")

Flagged tasks: 1016 / 8218 (12.4%)


## 3. Summary

In [5]:
if df.empty:
    print("No flagged tasks — safe to set allow_internet=false on all tasks.")
else:
    print("=== Confidence breakdown ===")
    print(df['confidence'].value_counts().to_string())
    print()
    print("=== Top categories in flagged tasks ===")
    print(df['category'].value_counts().head(20).to_string())
    print()
    print("=== Most common signal reasons ===")
    from collections import Counter
    reason_counts = Counter()
    for r in df['reasons']:
        for part in r.split('; '):
            reason_counts[part.strip()] += 1
    for reason, count in reason_counts.most_common(20):
        print(f"  {count:4d}  {reason}")

=== Confidence breakdown ===
confidence
high      552
medium    464

=== Top categories in flagged tasks ===
category
API testing and curl operations               93
package management                            47
certificate management                        47
pip package environment management            39
network diagnostics                           34
git repository operations                     33
git submodule management                      30
checksum verification                         28
launch a webserver                            27
headless browser data scraping                27
exploiting/fixing security vulnerabilities    27
semantic version bumping and changelogs       26
remote file synchronization                   25
firewall configuration                        24
JSON schema validation and jq processing      23
DNS and hostname resolution                   22
text diffing and patch application            20
Python virtual environment setup with venv    19


## 4. High-confidence flagged tasks (table)

In [6]:
high = df[df['confidence'] == 'high'].sort_values('n_high_signals', ascending=False)
print(f"High-confidence tasks: {len(high)}")
pd.set_option('display.max_colwidth', 120)
high[['task', 'category', 'reasons', 'first_snippet']]

High-confidence tasks: 552


,task,category,reasons,first_snippet
489,task_002800_91f2167a,remote file synchronization,fetch/pull from remote; rsync to remote host; git push/pull remote; sync from remote,"sync-based sync job pulls daily CSVs from a ""remote"" drop into /home/user/incoming and my script at /home/user/"
870,task_005825_4500a4c5,service configuration,pip install package (runtime); download from external; download (generic),it all afternoon. `pip install acmelib==2.3.1` against it fails with a hash mismatch — the
106,task_000390_a01ee4a6,package management,URL in instruction; pip install package (runtime),from the mirror at http://localhost:8080/simple/. I already updated the token in one p
225,task_001189_6a3cb7c7,launch a webserver,URL in instruction; curl to URL,".0:8080"", but `curl http://localhost:8080/health` just hangs and eventually times out."
180,task_000807_5c913149,API testing and curl operations,URL in instruction; curl to URL; fetch (generic),"1, but when I `curl http://127.0.0.1:8080/artifacts/web-api/latest` I get a 404 half th"
...,...,...,...,...
352,task_002211_d8e53282,binary data format parsing,URL in instruction,user-lookup API at http://localhost:8080/user/<uid> is returning garbage for some acco
349,task_002188_343a9988,certificate management,URL in instruction,Our test API on https://localhost:8443 is throwing cert errors when the mock client
345,task_002147_311d2f52,DNS and hostname resolution,fetch/pull from remote; fetch (generic),"and get the script fetching cleanly? The artifact server is a little mock thing that's already running on localhost,"
342,task_002124_862c0ba0,firewall configuration,git push/pull remote; remote server reference,"that's supposed to push run metadata to a ""remote"" mlflow-ish endpoint we've stubbed at 127.0.0.1:9090. Both"


## 5. Medium-confidence flagged tasks (table)

In [7]:
med = df[df['confidence'] == 'medium'].sort_values('n_medium_signals', ascending=False)
print(f"Medium-confidence tasks: {len(med)}")
med[['task', 'category', 'reasons', 'first_snippet']]

Medium-confidence tasks: 464


,task,category,reasons,first_snippet
737,task_004011_8f5ef38a,corrupt archive recovery,download (generic); fetch (generic),"arball's truncated (download got cut off) and `tar xzf` just bails with ""unexpected end"
1003,task_008060_4627a3f2,exploiting/fixing security vulnerabilities,download (generic); upload/post to service,"by messing with the download URL, and there's something sketchy about how it validates t"
61,task_000207_32e25167,CSV/JSON data manipulation,check connectivity; upload/post to service,flowing through? I checked and it's not a network thing.
683,task_003614_0f85f2b8,shell scripting automation,upload/post to service,"ision/hosts.csv and POST each host into the inventory API on localhost:8080, but the whole batch is coming back as fa"
682,task_003600_ef0b0bdd,Python virtual environment setup with venv,upload/post to service,pandas munging and posts summary rows to a little dashboard API running on localhost:8000. Worked fi
...,...,...,...,...
315,task_001928_1d83a94e,semantic version bumping and changelogs,upload/post to service,"hangelog entry, and POSTs to the little release-registry service on localhost:8817. Prob"
310,task_001896_6c1340ef,container management,upload/post to service,"ds batches of CSVs, posts each row to the local ingest service on :8080, and the service is suppo"
309,task_001893_6d053af6,API testing and curl operations,upload/post to service,Customer says our upload endpoint keeps 400ing on them but only for bigger payloads. I've got
307,task_001883_d700ba91,service configuration,upload/post to service,"up`), and compares. postgres is up, I can psql into it no problem. I haven't changed the manifests or the"


## 6. Full instructions for high-confidence tasks (manual review)

In [8]:
for _, row in high.iterrows():
    task_dir = DATASET_DIR / row['task']
    instruction = (task_dir / 'instruction.md').read_text(errors='replace')
    print(f"{'='*70}")
    print(f"TASK    : {row['task']}")
    print(f"CATEGORY: {row['category']}")
    print(f"REASONS : {row['reasons']}")
    print(f"\n{instruction}\n")

TASK    : task_002800_91f2167a
CATEGORY: remote file synchronization
REASONS : fetch/pull from remote; rsync to remote host; git push/pull remote; sync from remote

My rsync-based sync job pulls daily CSVs from a "remote" drop into /home/user/incoming and my script at /home/user/sync/merge.py stitches them into /home/user/sync/master.csv. Since Tuesday the merged file has fewer rows than the sum of the parts — like ~40k when I'd expect closer to 52k. No errors, exit 0, everything looks clean.

    I assumed dedup was eating them but the rows that vanish aren't dupes, they're legit distinct records. Files in incoming all look fine when I open them individually. Can you figure out where they're going and get master.csv whole again? The daily files themselves shouldn't be touched, ops re-pulls those.


TASK    : task_005825_4500a4c5
CATEGORY: service configuration
REASONS : pip install package (runtime); download from external; download (generic)

Our internal PyPI mirror at /home/user/py

## 7. Export findings to CSV

In [9]:
out_path = Path("/home/ec2-user/endless-terminals-playground/internet_scan_results.csv")
df.to_csv(out_path, index=False)
print(f"Saved to {out_path}")
print(f"\nSummary:")
print(f"  Total tasks scanned    : {len(task_dirs)}")
print(f"  Flagged (any)          : {len(df)}")
print(f"  High confidence        : {len(df[df.confidence=='high'])}")
print(f"  Medium confidence      : {len(df[df.confidence=='medium'])}")
print(f"  Likely safe (unflagged): {len(task_dirs) - len(df)}")

Saved to /home/ec2-user/endless-terminals-playground/internet_scan_results.csv

Summary:
  Total tasks scanned    : 8218
  Flagged (any)          : 1016
  High confidence        : 552
  Medium confidence      : 464
  Likely safe (unflagged): 7202
